# 01F — FINAL Independent Cross-Partition Verification

**NO TRAINING.**

Run only after 01E finishes.

This independently re-scans the FINAL dataset and checks:
- exact SHA-256 duplicates across partitions
- all pHash ≤12 candidate pairs across Train↔Validation, Train↔Test, Validation↔Test
- SIFT + RANSAC strict verification

Training begins only if this notebook prints:

`PASS ✅ FINAL CLEAN SPLIT VERIFIED.`

In [1]:
import sys,subprocess
subprocess.run([sys.executable,'-m','pip','install','-q','opencv-python-headless'],check=True)

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import hashlib,time
import cv2,numpy as np,pandas as pd

PROJECT=Path('/content/drive/MyDrive/Cataract')
ROOT=PROJECT/'Data_Clean_LeakageControlled_FINAL'
OUT=PROJECT/'FINAL_REVISION_2026_08'/'global_family_audit'/'final_independent_verification'
OUT.mkdir(parents=True,exist_ok=True)

CLASS_ORDER=['Cataract','Normal','Not Eye']
SPLITS=['Train','Validation','Test']
TH=12
assert ROOT.exists(),f'Run 01E first: {ROOT}'

Mounted at /content/drive


In [2]:
def pack64(bits):
    return np.packbits(bits.astype(np.uint8).reshape(-1)).tobytes().hex()

def phash(gray):
    small=cv2.resize(gray,(32,32),interpolation=cv2.INTER_AREA).astype(np.float32)
    D=cv2.dct(small)[:8,:8]
    med=np.median(D.flatten()[1:])
    return pack64(D.flatten()>med)

rows=[]
paths=[p for s in SPLITS for c in CLASS_ORDER for p in (ROOT/s/c).glob('*') if p.is_file()]
print('FINAL images:',len(paths))

for i,p in enumerate(paths,1):
    data=p.read_bytes()
    img=cv2.imdecode(np.frombuffer(data,np.uint8),cv2.IMREAD_COLOR)
    if img is None:continue
    gray=cv2.cvtColor(img,cv2.COLOR_BGR2GRAY)
    rel=p.relative_to(ROOT).as_posix()
    split,cls,fn=rel.split('/',2)
    rows.append({
        'path':rel,'split':split,'class':cls,'filename':fn,
        'sha256':hashlib.sha256(data).hexdigest(),
        'phash64':phash(gray)
    })
    if i%1000==0:print(i,'/',len(paths))

df=pd.DataFrame(rows)
df.to_csv(OUT/'final_fingerprints.csv',index=False)
print(df.groupby(['split','class']).size())

FINAL images: 13610
1000 / 13610
2000 / 13610
3000 / 13610
4000 / 13610
5000 / 13610
6000 / 13610
7000 / 13610
8000 / 13610
9000 / 13610
10000 / 13610
11000 / 13610
12000 / 13610
13000 / 13610
split       class   
Test        Cataract     854
            Normal       977
            Not Eye      756
Train       Cataract    2921
            Normal      3341
            Not Eye     2583
Validation  Cataract     719
            Normal       823
            Not Eye      636
dtype: int64


In [3]:
# exact cross-partition
bad=[]
for sha,g in df.groupby('sha256'):
    if g.split.nunique()>1:
        bad.append(g)
cross_exact=pd.concat(bad,ignore_index=True) if bad else pd.DataFrame(columns=df.columns)
cross_exact.to_csv(OUT/'FINAL_cross_partition_exact_duplicates.csv',index=False)
print('Exact cross-partition duplicate rows:',len(cross_exact))

Exact cross-partition duplicate rows: 0


## Exhaustive cross-partition pHash candidate search

Unlike the earlier nearest-neighbour audit, every pair across each split pair is considered.

In [4]:
LUT=np.array([bin(i).count('1') for i in range(256)],dtype=np.uint8)
def u64(s):return np.uint64(int(s,16))
def dist_block(A,B):
    x=np.bitwise_xor(A[:,None],B[None,:])
    by=x.view(np.uint8).reshape(x.shape+(8,))
    return LUT[by].sum(axis=-1)

df['ph_u']=df.phash64.map(u64)

split_pairs=[('Train','Validation'),('Train','Test'),('Validation','Test')]
records=[]

for sa,sb in split_pairs:
    A=df[df.split==sa].reset_index(drop=True)
    B=df[df.split==sb].reset_index(drop=True)
    bph=B.ph_u.to_numpy(np.uint64)

    print('\nScanning',sa,'vs',sb,len(A),'x',len(B))
    for start in range(0,len(A),128):
        end=min(start+128,len(A))
        D=dist_block(A.ph_u.to_numpy(np.uint64)[start:end],bph)
        ii,jj=np.where(D<=TH)
        for x,y in zip(ii,jj):
            ra=A.iloc[start+int(x)]
            rb=B.iloc[int(y)]
            records.append({
                'path_a':ra.path,'split_a':sa,'class_a':ra['class'],
                'path_b':rb.path,'split_b':sb,'class_b':rb['class'],
                'phash_distance':int(D[x,y])
            })
        if start%1280==0:
            print(end,'/',len(A),'candidate rows so far=',len(records))

cand=pd.DataFrame(records)
cand.to_csv(OUT/'FINAL_all_cross_partition_phash_candidates.csv',index=False)
print('\nTotal cross-partition pHash<=12 candidate pairs:',len(cand))


Scanning Train vs Validation 8845 x 2178
128 / 8845 candidate rows so far= 1
1408 / 8845 candidate rows so far= 55
2688 / 8845 candidate rows so far= 106
3968 / 8845 candidate rows so far= 159
5248 / 8845 candidate rows so far= 219
6528 / 8845 candidate rows so far= 260
7808 / 8845 candidate rows so far= 283

Scanning Train vs Test 8845 x 2587
128 / 8845 candidate rows so far= 299
1408 / 8845 candidate rows so far= 359
2688 / 8845 candidate rows so far= 410
3968 / 8845 candidate rows so far= 484
5248 / 8845 candidate rows so far= 565
6528 / 8845 candidate rows so far= 632
7808 / 8845 candidate rows so far= 654

Scanning Validation vs Test 2178 x 2587
128 / 2178 candidate rows so far= 677
1408 / 2178 candidate rows so far= 749

Total cross-partition pHash<=12 candidate pairs: 767


In [5]:
def prep(rel):
    im=cv2.imread(str(ROOT/rel),cv2.IMREAD_GRAYSCALE)
    h,w=im.shape
    sc=min(1.0,512/max(h,w))
    if sc<1:
        im=cv2.resize(im,(max(1,int(w*sc)),max(1,int(h*sc))),interpolation=cv2.INTER_AREA)
    return im

def verify(r):
    a,b=prep(r.path_a),prep(r.path_b)
    sift=cv2.SIFT_create(nfeatures=1200,contrastThreshold=0.02)
    k1,x1=sift.detectAndCompute(a,None)
    k2,x2=sift.detectAndCompute(b,None)
    good=[];inl=0;ratio=0.0
    if x1 is not None and x2 is not None and len(x1)>=2 and len(x2)>=2:
        bf=cv2.BFMatcher(cv2.NORM_L2)
        knn=bf.knnMatch(x1,x2,k=2)
        good=[p for p,q in knn if p.distance<0.75*q.distance]
        if len(good)>=4:
            src=np.float32([k1[m.queryIdx].pt for m in good]).reshape(-1,1,2)
            dst=np.float32([k2[m.trainIdx].pt for m in good]).reshape(-1,1,2)
            H,mask=cv2.findHomography(src,dst,cv2.RANSAC,5.0)
            if mask is not None:
                inl=int(mask.sum());ratio=inl/len(good)

    d=dict(r._asdict())
    d.update({
        'sift_good':len(good),'sift_inliers':inl,
        'sift_inlier_ratio':ratio,
        'confirmed_strict':bool(len(good)>=20 and inl>=15 and ratio>=0.5)
    })
    return d

VER=OUT/'FINAL_cross_partition_sift_verified.csv'
if VER.exists():
    old=pd.read_csv(VER)
    verified=old.to_dict('records')
    done=set(zip(old.path_a,old.path_b))
else:
    verified=[];done=set()

pending=[r for r in cand.itertuples(index=False) if (r.path_a,r.path_b) not in done]
print('SIFT pending:',len(pending))

for i,r in enumerate(pending,1):
    verified.append(verify(r))
    if i%250==0:
        pd.DataFrame(verified).to_csv(VER,index=False)
        print(i,'/',len(pending))

ver=pd.DataFrame(verified)
ver.to_csv(VER,index=False)

confirmed=ver[ver.confirmed_strict==True] if len(ver) else ver
confirmed.to_csv(OUT/'FINAL_CONFIRMED_CROSS_PARTITION_NEAR_DUPLICATES.csv',index=False)

print('\nConfirmed strict cross-partition near-duplicate pairs:',len(confirmed))

if len(cross_exact)==0 and len(confirmed)==0:
    print('\nPASS ✅ FINAL CLEAN SPLIT VERIFIED. You may begin model training.')
else:
    print('\nSTOP ❌ Do not train. Send this executed notebook back for review.')

SIFT pending: 767
250 / 767
500 / 767
750 / 767

Confirmed strict cross-partition near-duplicate pairs: 0

PASS ✅ FINAL CLEAN SPLIT VERIFIED. You may begin model training.
